# Notebook 4: CI/CD — El Quality Gate Automatizado

## Objetivos
- Ejecutar la suite de evaluación como un pipeline automatizable
- Definir thresholds de calidad (pass rate, scores mínimos)
- Implementar un gate pass/fail que bloquea el deploy si las evaluaciones fallan
- Entender cómo integrar esto en GitHub Actions
- Practicar el flujo: cambiar prompt → evaluar → promover o bloquear

## Contexto
En el Notebook 3 diseñamos test cases y aprendimos a evaluar con determinística y LLM-as-judge. Ahora convertimos eso en un **pipeline automatizado**: si cambia el código o el prompt, la evaluación se ejecuta automáticamente y decide si el cambio puede mergearse o no.

> **Concepto:** En CI/CD clásico, los tests unitarios bloquean el merge si fallan. Aquí hacemos lo mismo con evaluaciones del LLM.

## 1. Setup

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

from langfuse import Langfuse
import boto3

langfuse = Langfuse()
langfuse.auth_check()
print("[OK] Conectado a Langfuse")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=os.getenv("AWS_REGION", "eu-west-1"))
print("[OK] Bedrock runtime disponible")

In [ ]:
from techshop_agent import create_agent

agent = create_agent()
print("[OK] Agente TechShop cargado")

---

## Parte 0: Teoría — CI/CD para LLMs

### Qué es diferente

En CI/CD clásico, un test es determinístico: `assert 2 + 2 == 4` siempre pasa o siempre falla. En CI/CD para LLMs:

| Dimensión | CI/CD clásico | CI/CD para LLMs |
|-----------|--------------|------------------|
| **Tests** | Determinísticos | Pueden ser probabilísticos |
| **Resultado** | Pass/fail binario | Score con threshold (4.2/5 >= 4.0?) |
| **Coste** | Gratis | LLM-as-judge cuesta tokens |
| **Velocidad** | Segundos | Minutos (llamadas al LLM) |
| **Reproducibilidad** | 100% | Estocástico (misma query, respuesta diferente) |

### El pipeline que vamos a construir

```mermaid
flowchart TD
    T["Trigger: cambio en código o prompt"] --> S1["Stage 1: Eval determinística\n10 test cases, contains/regex\nGratis, menos de 1 min"]
    S1 --> S2["Stage 2: Eval LLM-as-judge\n5 test cases, relevancia/fidelidad\nCuesta tokens, menos de 5 min"]
    S2 --> G{"Gate:\npass rate >= 80%?"}
    G -->|Sí| D["Merge + Promote prompt"]
    G -->|No| B["PR bloqueado + Reporte"]
```

> **Implicación práctica:** Necesitas thresholds más flexibles que en testing clásico. Un 80% pass rate puede ser aceptable en lugar del 100% habitual.

---

## 2. Definir el dataset de evaluación

Usamos los mismos test cases del Notebook 3, pero ahora con thresholds explícitos.

In [ ]:
# Dataset de evaluacion con criterios de pass/fail
EVAL_DATASET = [
    # --- Determinística: el agente DEBE contener ciertos keywords ---
    {
        "query": "¿Qué portátiles tenéis?",
        "eval_type": "deterministic",
        "expected_contains": ["ProBook"],
        "expected_not_contains": ["MacBook", "Lenovo"],
    },
    {
        "query": "¿Cuánto cuesta el VoltPhone S?",
        "eval_type": "deterministic",
        "expected_contains": ["749"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Cuál es la política de devoluciones?",
        "eval_type": "deterministic",
        "expected_contains": ["30"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Hacéis envíos internacionales?",
        "eval_type": "deterministic",
        "expected_contains": ["no"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Quién es el presidente de Francia?",
        "eval_type": "deterministic",
        "expected_contains": [],
        "expected_not_contains": [],
        "should_reject": True,
    },
    # --- LLM-as-judge: calidad semántica ---
    {
        "query": "Necesito auriculares buenos pa música",
        "eval_type": "llm_judge",
        "criteria": "La respuesta recomienda productos del catálogo de TechShop y es profesional",
        "min_score": 3,
    },
    {
        "query": "¿Qué smartphone me recomendáis para hacer fotos?",
        "eval_type": "llm_judge",
        "criteria": "La respuesta menciona smartphones del catálogo de TechShop con sus características relevantes",
        "min_score": 3,
    },
    {
        "query": "Quiero comprar algo pero no sé qué, ¿qué tenéis?",
        "eval_type": "llm_judge",
        "criteria": "La respuesta es útil, menciona categorías disponibles y es profesional",
        "min_score": 3,
    },
]

print(f"[OK] Dataset: {len(EVAL_DATASET)} test cases")
print(f"  - Determinísticos: {sum(1 for t in EVAL_DATASET if t['eval_type'] == 'deterministic')}")
print(f"  - LLM-as-judge: {sum(1 for t in EVAL_DATASET if t['eval_type'] == 'llm_judge')}")

---

## 3. Funciones de evaluación

In [ ]:
def eval_deterministic(response: str, test_case: dict) -> dict:
    """Evaluación determinística: contains, not-contains, rejection."""
    passed = True
    details = []

    # Si debe rechazar la query (fuera de ámbito)
    if test_case.get("should_reject"):
        rejection_keywords = ["no puedo", "solo puedo", "fuera de", "no estoy"]
        has_rejection = any(kw in response.lower() for kw in rejection_keywords)
        if has_rejection:
            details.append("Rechazo detectado correctamente")
        else:
            passed = False
            details.append("Debería haber rechazado la query pero no lo hizo")
        return {"passed": passed, "details": details}

    # Verificar contains
    for keyword in test_case.get("expected_contains", []):
        if keyword.lower() in response.lower():
            details.append(f"Contiene '{keyword}'")
        else:
            passed = False
            details.append(f"NO contiene '{keyword}'")

    # Verificar not-contains
    for keyword in test_case.get("expected_not_contains", []):
        if keyword.lower() in response.lower():
            passed = False
            details.append(f"Contiene '{keyword}' (NO debería)")
        else:
            details.append(f"No contiene '{keyword}' (correcto)")

    return {"passed": passed, "details": details}


def eval_llm_judge(query: str, response: str, criteria: str, min_score: int = 3) -> dict:
    """Evaluación con LLM-as-judge usando Bedrock."""
    judge_prompt = f"""Evalúa la siguiente respuesta de un agente de customer service.

Consulta del usuario: {query}
Respuesta del agente: {response}

Criterio de evaluación: {criteria}

Responde SOLO con JSON válido:
{{"score": <1-5>, "justification": "<explicación breve>"}}

Donde 1 = muy malo, 5 = excelente."""

    result = bedrock_runtime.converse(
        modelId=os.getenv("MODEL_ID", "eu.anthropic.claude-haiku-4-5-20251001-v1:0"),
        messages=[{"role": "user", "content": [{"text": judge_prompt}]}],
        inferenceConfig={"maxTokens": 200, "temperature": 0.0},
    )

    response_text = result["output"]["message"]["content"][0]["text"]
    try:
        judgment = json.loads(response_text)
        passed = judgment.get("score", 0) >= min_score
        return {
            "passed": passed,
            "score": judgment.get("score", 0),
            "justification": judgment.get("justification", ""),
        }
    except json.JSONDecodeError:
        return {"passed": False, "score": 0, "justification": f"JSON inválido: {response_text[:100]}"}


print("[OK] Funciones de evaluación definidas")

---

## 4. Ejecutar el pipeline completo

In [ ]:
# --- PIPELINE DE EVALUACION ---
PASS_RATE_THRESHOLD = 0.8  # 80% mínimo para pasar
results = []

print("=" * 70)
print("  PIPELINE DE EVALUACION CI/CD")
print("=" * 70)

for i, test_case in enumerate(EVAL_DATASET):
    query = test_case["query"]
    print(f"\n[{i+1}/{len(EVAL_DATASET)}] {query[:60]}...")

    # Ejecutar agente
    response = str(agent(query))

    # Evaluar según tipo
    if test_case["eval_type"] == "deterministic":
        result = eval_deterministic(response, test_case)
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] Determinística: {', '.join(result['details'][:2])}")
    else:
        result = eval_llm_judge(query, response, test_case["criteria"], test_case.get("min_score", 3))
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] LLM-as-judge: score={result.get('score', '?')}/5 | {result.get('justification', '')[:60]}")

    results.append({
        "query": query,
        "eval_type": test_case["eval_type"],
        "passed": result["passed"],
        "details": result,
    })

# --- RESULTADO DEL GATE ---
total = len(results)
passed = sum(1 for r in results if r["passed"])
pass_rate = passed / total if total > 0 else 0

print("\n" + "=" * 70)
print(f"  RESULTADO: {passed}/{total} pasaron ({pass_rate:.0%})")
print(f"  THRESHOLD: {PASS_RATE_THRESHOLD:.0%}")
print(f"  GATE: {'PASS -- merge permitido' if pass_rate >= PASS_RATE_THRESHOLD else 'FAIL -- PR bloqueado'}")
print("=" * 70)

# Detalle de fallos
failed = [r for r in results if not r["passed"]]
if failed:
    print(f"\nTest cases que fallaron ({len(failed)}):")
    for r in failed:
        print(f"  - [{r['eval_type']}] {r['query'][:50]}...")

---

## 5. Prompt deploy automatizado

Si el gate pasa, el siguiente paso es **promover el prompt a producción**. Esto se hace moviendo el label `production` en Langfuse.

In [ ]:
# Simular el flujo de prompt deploy
# En un CI/CD real, esto se ejecutaría automáticamente si el gate pasa

if pass_rate >= PASS_RATE_THRESHOLD:
    print("[OK] Gate PASS -- flujo de deploy:")
    print("  1. Evaluaciones pasaron")
    print("  2. Mover label 'production' al prompt evaluado")
    print("  3. El agente en producción lee el nuevo prompt automáticamente")
    print("  4. Langfuse tracing monitoriza el comportamiento")
    print()
    print("  # Código para promote (ejecutar en CI/CD):")
    print("  # langfuse.update_prompt_label('techshop-system-prompt', version=N, label='production')")
else:
    print("[ERROR] Gate FAIL -- acciones:")
    print("  1. PR bloqueado, no se puede mergear")
    print("  2. Revisar test cases que fallaron")
    print("  3. Ajustar prompt o código y volver a ejecutar")

---

## 6. Cómo integrar en GitHub Actions

El pipeline que acabamos de ejecutar manualmente se puede automatizar en CI/CD:

```yaml
# .github/workflows/eval.yml
name: LLM Evaluation Gate

on:
  pull_request:
    paths:
      - "prompts/**"
      - "src/techshop_agent/**"

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -e ".[dev]"
      - name: Run evaluation pipeline
        run: python scripts/eval_pipeline.py
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
```

El script `eval_pipeline.py` contiene exactamente lo que ejecutamos en la celda anterior, con `sys.exit(1)` si el gate falla.

> **El código de este notebook ES el script de CI/CD.** Solo hay que moverlo de notebook a script Python.

---

## Ejercicios opcionales

### Ejercicio 1: Añadir test cases
Añade 3 test cases más al dataset:
- Uno para un producto específico con precio exacto
- Uno adversarial (prompt injection)
- Uno edge case (typos, idioma mixto)

### Ejercicio 2: Ajustar el threshold
Cambia `PASS_RATE_THRESHOLD` a 0.9 (90%). ¿Sigue pasando? ¿Qué test cases deberían mejorar?

### Ejercicio 3: Prompt deploy automático
Implementa el promote real: si el gate pasa, mueve el label `production` al prompt v2 en Langfuse.

## Resumen

| Concepto | Qué aprendimos |
|----------|----------------|
| **Pipeline CI/CD** | Checks + eval determinística + LLM-as-judge en secuencia |
| **Quality gate** | Pass rate >= 80% para permitir merge |
| **Prompt deploy** | Labels de Langfuse, zero-downtime, rollback instantáneo |
| **Automatización** | El pipeline del notebook se convierte en script de CI/CD |

### Roadmap del curso

```mermaid
flowchart LR
    NB1["NB 1\nSetup y Agente"] --> NB2["NB 2\nObservabilidad"]
    NB2 --> NB3["NB 3\nPrompt Management"]
    NB3 --> NB4["NB 4\nEvaluación"]
    NB4 --> NB5["NB 5 (HOY)\nCI/CD Gate"]
    NB5 --> NB6["NB 6\nGuardrails"]
    NB6 --> NB7["NB 7\nPipeline Integrado"]

    style NB5 fill:#2d5016,stroke:#4a8c28,color:#fff
```

> **Concepto clave:** Del "deployamos y rezamos" al "deployamos cuando pasa los tests".

---

### Referencias

- [Langfuse — Datasets](https://langfuse.com/docs/datasets)
- [Langfuse — Prompt Management](https://langfuse.com/docs/prompts/get-started)
- [promptfoo — CI/CD Integration](https://www.promptfoo.dev/docs/integrations/github-action/)
- [deepeval — Testing LLMs](https://docs.confident-ai.com/)
- [Hamel Husain — Your AI Product Needs Evals](https://hamel.dev/blog/posts/evals/)

---

## Siguiente: [Notebook 5 — Guardrails con Bedrock](../day_3/05_guardrails.ipynb)

# Notebook 4: CI/CD -- El Quality Gate Automatizado

## Objetivos
- Ejecutar la suite de evaluacion como un pipeline automatizable
- Definir thresholds de calidad (pass rate, scores minimos)
- Implementar un gate pass/fail que bloquea el deploy si las evaluaciones fallan
- Entender como integrar esto en GitHub Actions
- Practicar el flujo: cambiar prompt -> evaluar -> promover o bloquear

## Contexto
En el Notebook 3 diseñamos test cases y aprendimos a evaluar con deterministica y LLM-as-judge. Ahora convertimos eso en un **pipeline automatizado**: si cambia el codigo o el prompt, la evaluacion se ejecuta automaticamente y decide si el cambio puede mergearse o no.

> **Concepto:** En CI/CD clasico, los tests unitarios bloquean el merge si fallan. Aqui hacemos lo mismo con evaluaciones del LLM.

## 1. Setup

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

from langfuse import Langfuse, observe, langfuse_context
import boto3

langfuse = Langfuse()
langfuse.auth_check()
print("[OK] Conectado a Langfuse")

bedrock_runtime = boto3.client("bedrock-runtime", region_name=os.getenv("AWS_REGION", "eu-west-1"))
print("[OK] Bedrock runtime disponible")

In [ ]:
from techshop_agent import create_agent

agent = create_agent()
print("[OK] Agente TechShop cargado")

---

## Parte 0: Teoria -- CI/CD para LLMs

### ¿Que es diferente?

En CI/CD clasico, un test es deterministico: `assert 2 + 2 == 4` siempre pasa o siempre falla. En CI/CD para LLMs:

| Dimension | CI/CD clasico | CI/CD para LLMs |
|-----------|--------------|-----------------|
| **Tests** | Deterministicos | Pueden ser probabilisticos |
| **Resultado** | Pass/fail binario | Score con threshold (4.2/5 >= 4.0?) |
| **Coste** | Gratis | LLM-as-judge cuesta tokens |
| **Velocidad** | Segundos | Minutos (llamadas al LLM) |
| **Reproducibilidad** | 100% | Estocastico (misma query, respuesta diferente) |

### El pipeline que vamos a construir

```mermaid
flowchart TD
    T["Trigger: cambio en codigo o prompt"] --> S1["Stage 1: Eval deterministica<br/>10 test cases, contains/regex<br/>Gratis, menos de 1 min"]
    S1 --> S2["Stage 2: Eval LLM-as-judge<br/>5 test cases, relevancia/fidelidad<br/>Cuesta tokens, menos de 5 min"]
    S2 --> G{"Gate:<br/>pass rate >= 80%?"}
    G -->|Si| D["Merge + Promote prompt"]
    G -->|No| B["PR bloqueado + Reporte"]
```

> **Implicacion practica:** Necesitas thresholds mas flexibles que en testing clasico. Un 80% pass rate puede ser aceptable en lugar del 100% habitual.

---

## 2. Definir el dataset de evaluacion

Usamos los mismos test cases del Notebook 3, pero ahora con thresholds explicitos.

In [ ]:
# Dataset de evaluacion con criterios de pass/fail
EVAL_DATASET = [
    # --- Deterministica: el agente DEBE contener ciertos keywords ---
    {
        "query": "¿Que portatiles teneis?",
        "eval_type": "deterministic",
        "expected_contains": ["ProBook"],
        "expected_not_contains": ["MacBook", "Lenovo"],
    },
    {
        "query": "¿Cuanto cuesta el VoltPhone S?",
        "eval_type": "deterministic",
        "expected_contains": ["749"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Cual es la politica de devoluciones?",
        "eval_type": "deterministic",
        "expected_contains": ["30"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Haceis envios internacionales?",
        "eval_type": "deterministic",
        "expected_contains": ["no"],
        "expected_not_contains": [],
    },
    {
        "query": "¿Quien es el presidente de Francia?",
        "eval_type": "deterministic",
        "expected_contains": [],
        "expected_not_contains": [],
        "should_reject": True,
    },
    # --- LLM-as-judge: calidad semantica ---
    {
        "query": "Necesito auriculares buenos pa musica",
        "eval_type": "llm_judge",
        "criteria": "La respuesta recomienda productos del catalogo de TechShop y es profesional",
        "min_score": 3,
    },
    {
        "query": "¿Que smartphone me recomendais para hacer fotos?",
        "eval_type": "llm_judge",
        "criteria": "La respuesta menciona smartphones del catalogo de TechShop con sus caracteristicas relevantes",
        "min_score": 3,
    },
    {
        "query": "Quiero comprar algo pero no se que, ¿que teneis?",
        "eval_type": "llm_judge",
        "criteria": "La respuesta es util, menciona categorias disponibles y es profesional",
        "min_score": 3,
    },
]

print(f"[OK] Dataset: {len(EVAL_DATASET)} test cases")
print(f"  - Deterministicos: {sum(1 for t in EVAL_DATASET if t['eval_type'] == 'deterministic')}")
print(f"  - LLM-as-judge: {sum(1 for t in EVAL_DATASET if t['eval_type'] == 'llm_judge')}")

---

## 3. Funciones de evaluacion

In [ ]:
def eval_deterministic(response: str, test_case: dict) -> dict:
    """Evaluacion deterministica: contains, not-contains, rejection."""
    passed = True
    details = []

    # Si debe rechazar la query (fuera de ambito)
    if test_case.get("should_reject"):
        rejection_keywords = ["no puedo", "solo puedo", "fuera de", "no estoy"]
        has_rejection = any(kw in response.lower() for kw in rejection_keywords)
        if has_rejection:
            details.append("Rechazo detectado correctamente")
        else:
            passed = False
            details.append("Deberia haber rechazado la query pero no lo hizo")
        return {"passed": passed, "details": details}

    # Verificar contains
    for keyword in test_case.get("expected_contains", []):
        if keyword.lower() in response.lower():
            details.append(f"Contiene '{keyword}'")
        else:
            passed = False
            details.append(f"NO contiene '{keyword}'")

    # Verificar not-contains
    for keyword in test_case.get("expected_not_contains", []):
        if keyword.lower() in response.lower():
            passed = False
            details.append(f"Contiene '{keyword}' (NO deberia)")
        else:
            details.append(f"No contiene '{keyword}' (correcto)")

    return {"passed": passed, "details": details}


def eval_llm_judge(query: str, response: str, criteria: str, min_score: int = 3) -> dict:
    """Evaluacion con LLM-as-judge usando Bedrock."""
    judge_prompt = f"""Evalua la siguiente respuesta de un agente de customer service.

Consulta del usuario: {query}
Respuesta del agente: {response}

Criterio de evaluacion: {criteria}

Responde SOLO con JSON valido:
{{"score": <1-5>, "justification": "<explicacion breve>"}}

Donde 1 = muy malo, 5 = excelente."""

    result = bedrock_runtime.converse(
        modelId=os.getenv("MODEL_ID", "eu.anthropic.claude-haiku-4-5-20251001-v1:0"),
        messages=[{"role": "user", "content": [{"text": judge_prompt}]}],
        inferenceConfig={"maxTokens": 200, "temperature": 0.0},
    )

    response_text = result["output"]["message"]["content"][0]["text"]
    try:
        judgment = json.loads(response_text)
        passed = judgment.get("score", 0) >= min_score
        return {
            "passed": passed,
            "score": judgment.get("score", 0),
            "justification": judgment.get("justification", ""),
        }
    except json.JSONDecodeError:
        return {"passed": False, "score": 0, "justification": f"JSON invalido: {response_text[:100]}"}


print("[OK] Funciones de evaluacion definidas")

---

## 4. Ejecutar el pipeline completo

In [ ]:
# --- PIPELINE DE EVALUACION ---
PASS_RATE_THRESHOLD = 0.8  # 80% minimo para pasar
results = []

print("=" * 70)
print("  PIPELINE DE EVALUACION CI/CD")
print("=" * 70)

for i, test_case in enumerate(EVAL_DATASET):
    query = test_case["query"]
    print(f"\n[{i+1}/{len(EVAL_DATASET)}] {query[:60]}...")

    # Ejecutar agente
    response = str(agent(query))

    # Evaluar segun tipo
    if test_case["eval_type"] == "deterministic":
        result = eval_deterministic(response, test_case)
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] Deterministica: {', '.join(result['details'][:2])}")
    else:
        result = eval_llm_judge(query, response, test_case["criteria"], test_case.get("min_score", 3))
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] LLM-as-judge: score={result.get('score', '?')}/5 | {result.get('justification', '')[:60]}")

    results.append({
        "query": query,
        "eval_type": test_case["eval_type"],
        "passed": result["passed"],
        "details": result,
    })

# --- RESULTADO DEL GATE ---
total = len(results)
passed = sum(1 for r in results if r["passed"])
pass_rate = passed / total if total > 0 else 0

print("\n" + "=" * 70)
print(f"  RESULTADO: {passed}/{total} pasaron ({pass_rate:.0%})")
print(f"  THRESHOLD: {PASS_RATE_THRESHOLD:.0%}")
print(f"  GATE: {'PASS -- merge permitido' if pass_rate >= PASS_RATE_THRESHOLD else 'FAIL -- PR bloqueado'}")
print("=" * 70)

# Detalle de fallos
failed = [r for r in results if not r["passed"]]
if failed:
    print(f"\nTest cases que fallaron ({len(failed)}):")
    for r in failed:
        print(f"  - [{r['eval_type']}] {r['query'][:50]}...")

---

## 5. Prompt deploy automatizado

Si el gate pasa, el siguiente paso es **promover el prompt a produccion**. Esto se hace moviendo el label `production` en Langfuse.

In [ ]:
# Simular el flujo de prompt deploy
# En un CI/CD real, esto se ejecutaria automaticamente si el gate pasa

if pass_rate >= PASS_RATE_THRESHOLD:
    print("[OK] Gate PASS -- flujo de deploy:")
    print("  1. Evaluaciones pasaron")
    print("  2. Mover label 'production' al prompt evaluado")
    print("  3. El agente en produccion lee el nuevo prompt automaticamente")
    print("  4. Langfuse tracing monitoriza el comportamiento")
    print()
    print("  # Codigo para promote (ejecutar en CI/CD):")
    print("  # langfuse.update_prompt_label('techshop-system-prompt', version=N, label='production')")
else:
    print("[ERROR] Gate FAIL -- acciones:")
    print("  1. PR bloqueado, no se puede mergear")
    print("  2. Revisar test cases que fallaron")
    print("  3. Ajustar prompt o codigo y volver a ejecutar")

---

## 6. Como integrar en GitHub Actions

El pipeline que acabamos de ejecutar manualmente se puede automatizar en CI/CD:

```yaml
# .github/workflows/eval.yml
name: LLM Evaluation Gate

on:
  pull_request:
    paths:
      - "prompts/**"
      - "src/techshop_agent/**"

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -e ".[dev]"
      - name: Run evaluation pipeline
        run: python scripts/eval_pipeline.py
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
```

El script `eval_pipeline.py` contiene exactamente lo que ejecutamos en la celda anterior, con `sys.exit(1)` si el gate falla.

> **El codigo de este notebook ES el script de CI/CD.** Solo hay que moverlo de notebook a script Python.

---

## Ejercicios opcionales

### Ejercicio 1: Anadir test cases
Anade 3 test cases mas al dataset:
- Uno para un producto especifico con precio exacto
- Uno adversarial (prompt injection)
- Uno edge case (typos, idioma mixto)

### Ejercicio 2: Ajustar el threshold
Cambia `PASS_RATE_THRESHOLD` a 0.9 (90%). ¿Sigue pasando? ¿Que test cases deberian mejorar?

### Ejercicio 3: Prompt deploy automatico
Implementa el promote real: si el gate pasa, mueve el label `production` al prompt v2 en Langfuse.

## Resumen

| Concepto | Que aprendimos |
|----------|----------------|
| **Pipeline CI/CD** | Checks + eval deterministica + LLM-as-judge en secuencia |
| **Quality gate** | Pass rate >= 80% para permitir merge |
| **Prompt deploy** | Labels de Langfuse, zero-downtime, rollback instantaneo |
| **Automatizacion** | El pipeline del notebook se convierte en script de CI/CD |

### Roadmap del curso

```mermaid
flowchart LR
    NB1["NB 1<br/>Setup y Agente"] --> NB2["NB 2<br/>Observabilidad"]
    NB2 --> NB3["NB 3<br/>Prompt Management"]
    NB3 --> NB4["NB 4<br/>Evaluacion"]
    NB4 --> NB5["NB 5 (HOY)<br/>CI/CD Gate"]
    NB5 --> NB6["NB 6<br/>Guardrails"]
    NB6 --> NB7["NB 7<br/>Pipeline Integrado"]

    style NB5 fill:#2d5016,stroke:#4a8c28,color:#fff
```

> **Concepto clave:** Del "deployamos y rezamos" al "deployamos cuando pasa los tests".

---

### Referencias

- [Langfuse -- Datasets](https://langfuse.com/docs/datasets)
- [Langfuse -- Prompt Management](https://langfuse.com/docs/prompts/get-started)
- [promptfoo -- CI/CD Integration](https://www.promptfoo.dev/docs/integrations/github-action/)
- [deepeval -- Testing LLMs](https://docs.confident-ai.com/)
- [Hamel Husain -- Your AI Product Needs Evals](https://hamel.dev/blog/posts/evals/)

---

## Siguiente: [Notebook 5 - Guardrails con Bedrock](../day_3/05_guardrails.ipynb)